# Chapter 1 — Introduction to Triton

Part 1: Foundations — what Triton is, how programs run, and how vectors are built.

**Thread.** One idea recurs in every section:

- A launch is a grid of independent programs.
- Each program owns one tile.
- A tile's indices are `pid*BLOCK + arange(BLOCK)`.
- The mask `offs < N` guards the tail.
- The host chooses shapes and grid; the device computes tiles.

- **Concept arc.** The ladder is **concept → numerical demo → micro-kernel → applied use**; you build the mental model, do one numerical demo by hand, write the smallest JIT kernel, then fuse two operations to see why tile programming pays off.

**Study tip.**

- Fix the spine: $N=8$, $\texttt{BLOCK}=4$, $x=(1,2,3,4,5,6,7,8)$, $y=10x$.
- Write the grid $P=\lceil N/\texttt{BLOCK}\rceil=2$ on paper.
- Write the two rows `pid -> offs -> mask` and the eight outputs.
- Repeat for $N=10$: program 2 has `offs = 8,9,10,11` and mask `T,T,F,F`.
- If your paper rows match the worked example, every later kernel is the same skeleton with one expression changed.

**Learning objectives**

- Explain what Triton is and which parts of a GPU program the compiler owns.
- Place allocation, grid choice, JIT specialization, and tile math on the correct side of the host/device boundary.
- Compute the grid $P=\lceil N/B\rceil$, the program id, the offsets `pid*BLOCK + arange(BLOCK)`, and the mask `offs < N` for a 1-D launch.
- Classify every kernel argument as compile-time (`tl.constexpr`) or runtime, and predict when a launch recompiles.
- Write and launch a minimal `@triton.jit` vector-add kernel with masked load and store.
- State the 1-D shape/argument contract: equal-length input buffers, power-of-two `BLOCK`, one program per tile, tail guarded by `offs < N`.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [Next →](../ch02_program_execution/02_complete.ipynb)

**Estimated time:** 45–60 min · **Difficulty:** Beginner

**Prerequisites:** Python functions and loops; basic PyTorch tensor allocation (`torch.empty`, `torch.arange`); no GPU required and no CUDA experience assumed.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)

## Chapter map

| § | Topic | Key idea/formula | Shape or argument rule | Kernel/demo |
|---|---|---|---|---|
| **1.1** | What is Triton? | A Python DSL whose unit of work is a tile: $Y=f(\text{tile}(X))$ | tile length = `BLOCK`; whole tensor length = `N` | tile lift of the spine |
| **1.2** | Why Triton instead of CUDA? | replace per-thread indexing with per-tile expressions | $P=\lceil N/B\rceil$ programs, each of $B$ lanes | thread count vs program count |
| **1.3** | Triton vs PyTorch vs CUDA | PyTorch: $Y=f(X)$; Triton: you write the kernel; CUDA: you write the thread | PyTorch broadcasts; Triton requires same-length buffers | three-layer comparison |
| **1.4** | Triton execution model | grid of programs $\to$ one CTA per program $\to$ tile of lanes | `pid*BLOCK + arange(BLOCK)`; mask the tail | numeric tile map |
| **1.5** | Host code vs device code | host: allocate + grid; device: tile math | host shapes are `(N,)`; device sees raw pointers | simulated host/device split |
| **1.6** | JIT compilation with `@triton.jit` | parse once, specialize on types + constexprs, cache | pointers/scalars runtime; `BLOCK` compile-time | signature inspection |
| **1.7** | Kernel launch syntax | `kernel[grid](args, BLOCK=...)` | $P=\lceil N/B\rceil$; one integer per grid axis | grid arithmetic |
| **1.8** | Compile-time vs runtime arguments | runtime changes data; constexpr changes the binary | `n_elements` runtime; `BLOCK` constexpr | argument classifier |
| **1.9** | `tl.constexpr` | value is known while compiling; loops and `arange` can use it | `BLOCK` must be a power of two for `tl.arange` | two `BLOCK` specializations |
| **1.10** | First working Triton kernel | $y_i=x_i+y_i$ per lane | same shape `(N,)`, masked tail | `add_kernel` |

- **Thesis.** You do not schedule threads; you describe a tile. The compiler maps the tile to threads, registers, and memory transactions, and the host decides how many programs exist.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import pytest
import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, check, require_gpu, run_if_gpu

# Chapter constants, asserted once so every later claim has a fixed reference.
N, BLOCK = 8, 4
spine = torch.arange(1, N + 1, dtype=torch.float32)
assert tuple(spine.shape) == (8,)
assert spine.dtype == torch.float32
assert spine.stride() == (1,)
assert (triton.cdiv(N, BLOCK),) == (2,)

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)

## Concept diagrams

- **Connection.** The opening fixed one vector and one block size; these diagrams show the same launch from three angles.

### Host / device boundary

```text
 HOST (ordinary Python)                         DEVICE (compiled kernel)
 ----------------------                         ------------------------
 allocate x, y, out          launch            program pid reads/writes
 choose grid P = ceil(N/B)  ---------->          offsets = pid*B + arange(B)
 choose BLOCK, dtype                             mask = offsets < N
 inspect / compare output   <----------          load -> compute -> store
```

- **Rule.** The left column runs once per launch; the right column runs once per program, concurrently, with no shared Python state.

### Compile time vs runtime

```text
 first launch of add_kernel[grid](x, y, out, N, BLOCK=4)
   |
   |  parse AST
   |  specialize: pointer types, scalar types, constexpr values
   |  compile + cache key = (signature, constexprs, arch)
   v
 binary for (fp32*, fp32*, fp32*, int32, BLOCK=4)

 second launch with a new N  -> cache hit, no compile
 second launch with BLOCK=8  -> different key, compile a second binary
```

- **Rule.** Runtime data reuses the binary; a changed `constexpr` or dtype creates a second binary.

### Grid -> program -> tile

```text
 N = 8, BLOCK = 4, P = 2

            program 0                    program 1
        +----------------+          +----------------+
 x      | 1  2  3  4      |          | 5  6  7  8      |
 y      |10 20 30 40      |          |50 60 70 80      |
        +----------------+          +----------------+
 out    |11 22 33 44      |          |55 66 77 88      |
        +----------------+          +----------------+
 offs     0  1  2  3                  4  5  6  7
 mask     T  T  T  T                  T  T  T  T
```

- **Rule.** Every lane belongs to exactly one program, and every program owns exactly `BLOCK` lanes.
- **Tail.** When $N$ is not a multiple of `BLOCK`, the last program owns lanes that do not exist, and the mask turns them off.
- **Bridge.** Part I turns these pictures into ten precise rules.

In [ ]:
N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)
y = 10.0 * x
grid = (triton.cdiv(N, BLOCK),)

assert tuple(x.shape) == (8,)
assert x.dtype == torch.float32
assert x.stride() == (1,)
assert tuple(y.shape) == (8,)
assert y.dtype == torch.float32
assert grid == (2,)

for pid in range(grid[0]):
    offs = pid * BLOCK + torch.arange(BLOCK)
    mask = offs < N
    assert tuple(offs.shape) == (4,)
    assert mask.dtype == torch.bool
    assert bool(mask.all())
    assert offs.tolist() == [pid * BLOCK + i for i in range(BLOCK)]
    assert torch.equal(
        x[offs],
        torch.arange(1 + pid * BLOCK, 1 + (pid + 1) * BLOCK, dtype=torch.float32),
    )
    assert tuple(x[offs][mask].shape) == (4,)

print("spine: shape (8,), dtype float32, grid (2,), masks all true")

# Part I — Concepts and mathematics

## 1.1 What is Triton?

- **Connection.** The opening fixed the spine and the tile size; this section names the language that operates on those tiles.
- **Rule.** Triton is a Python-embedded language and compiler for GPU kernels at the level of a **tile**, not a thread; a kernel is a Python function decorated with `@triton.jit`.
- **Formula.** For a pointwise map $f$ and tile length $B$, $Y_p=f(X_p)$ with $X_p=X[pB:pB+B]$ and $p=0,\ldots,P-1$.
- **Example.** On the spine, $X_0=(1,2,3,4)$ and $X_1=(5,6,7,8)$; the add kernel maps each tile to $11X_p$.
- **Where.** It is the shortest path from a mathematical expression to a custom GPU kernel while keeping Python syntax and PyTorch tensors.
- **Triton.** The unit of value inside a kernel is a tensor of shape `(BLOCK,)`; `tl.arange(0, BLOCK)` builds its index vector.
- **Failure.** Treating a kernel as a scalar function, with `for i in range(n)` or one-element indexing, throws away the tile model.
- **Bridge.** Section 1.2 contrasts this tile-level model with CUDA's thread-level model.

In [ ]:
N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)
tiles = x.reshape(N // BLOCK, BLOCK)

assert tuple(x.shape) == (8,)
assert tuple(tiles.shape) == (2, 4)
assert tuple(tiles[0].shape) == (4,)
assert tiles[0].tolist() == [1.0, 2.0, 3.0, 4.0]
assert tiles[1].tolist() == [5.0, 6.0, 7.0, 8.0]
assert tuple((tiles[0] + tiles[1]).shape) == (4,)
assert (tiles[0] + tiles[1]).tolist() == [6.0, 8.0, 10.0, 12.0]

## 1.2 Why Triton instead of CUDA?

- **Connection.** Section 1.1 defined a kernel over tiles; this section asks what changes when the same work is written in CUDA.
- **Rule.** CUDA exposes the thread; Triton exposes the tile. A CUDA add computes a global thread index, checks it against $N$, and handles one element; a Triton add computes a program id, builds `BLOCK` offsets, masks the tail once, and applies one expression to every lane.
- **Formula.** CUDA launches $N$ threads; Triton launches $P=\lceil N/B\rceil$ programs with $B$ lanes each.
- **Example.** For $N=8$, $B=4$: 8 CUDA threads with 8 bounds checks versus 2 Triton programs with one mask per program.
- **Where.** The compiler chooses the thread count, vector width, and register layout, so one source travels across NVIDIA and AMD GPUs.
- **Triton.** Write one expression for the whole tile; use a mask when a lane may not exist, never a per-lane `if`.
- **Failure.** Porting `tid = blockIdx.x * blockDim.x + threadIdx.x` literally; Triton has no `threadIdx`.
- **Bridge.** Section 1.3 places PyTorch, Triton, and CUDA side by side.

In [ ]:
N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)

cuda_threads = N
triton_programs = triton.cdiv(N, BLOCK)

assert tuple(x.shape) == (8,)
assert x.numel() == N
assert cuda_threads == 8
assert triton_programs == 2
assert triton_programs * BLOCK == N
assert tuple((triton_programs, BLOCK)) == (2, 4)

## 1.3 Triton vs PyTorch vs CUDA

- **Connection.** Section 1.2 separated thread-level CUDA from tile-level Triton; this section adds the third layer, PyTorch.
- **Rule.** Choose the lowest layer that exposes the control you need.

| Layer | You write | The runtime gives you | Typical use |
|---|---|---|---|
| **PyTorch** | `out = x + y` | a prebuilt, autotuned kernel + autograd | models, prototyping |
| **Triton** | a `@triton.jit` tile kernel + grid | JIT specialization, layout, vectorization | fused or missing ops |
| **CUDA** | a thread kernel + block/grid dims | full control of threads, shared memory, sync | peak-performance libraries |

- **Formula.** PyTorch runs $Y=f(X)$; Triton runs $Y=f(\text{tile})$; CUDA runs $y_i=f(x_i)$ per thread.
- **Example.** The vector add is `x + y` in PyTorch, one tile kernel in Triton, and one thread per element in CUDA.
- **Where.** Use PyTorch until it fuses the wrong thing or lacks the op; use Triton for fusion or custom memory patterns without warp-level control; use CUDA for shared memory, `__syncthreads`, or hand-tuned tensor cores.
- **Triton.** A Triton kernel is launched from PyTorch and returns an ordinary tensor, so it composes with the model.
- **Failure.** Assuming Triton replaces PyTorch; it complements allocation, autograd, and the module system.
- **Bridge.** Section 1.4 defines the grid that a Triton launch creates.

In [ ]:
N = 8
x = torch.arange(1, N + 1, dtype=torch.float32)
y = 10.0 * x
torch_out = x + y

assert tuple(torch_out.shape) == (8,)
assert torch_out.dtype == torch.float32
assert torch_out.stride() == (1,)
assert torch_out.tolist() == [11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0]
assert torch.equal(torch_out, 11.0 * x)

## 1.4 Triton execution model

- **Connection.** Section 1.3 said Triton launches a tile kernel; this section defines the grid that launch creates.
- **Rule.** A launch creates a grid of independent programs. Program `pid = tl.program_id(0)` is one CTA and owns one tile.
- **Formula.** Program $p$ owns $\text{offs}_p=pB+\{0,\ldots,B-1\}$, $\text{mask}_p=\text{offs}_p\lt N$, and $P=\lceil N/B\rceil$.
- **Example.** On the spine, `pid=0` owns `offs=0,1,2,3` and `pid=1` owns `offs=4,5,6,7`; both masks are all true.
- **Where.** This is the mental model for matrix tiles, row/column programs, reductions, and multi-kernel pipelines.
- **Triton.** `tl.program_id(0)` is the only coordinate of a 1-D launch; the device does not see $P$ unless you pass it or call `tl.num_programs(0)`.
- **Failure.** Forgetting the tail mask, or launching $P=\lfloor N/B\rfloor$ programs and dropping the tail.
- **Bridge.** Section 1.5 splits the host and device sides of this launch.

In [ ]:
N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)
P = triton.cdiv(N, BLOCK)

assert P == 2
assert (P,) == (2,)
tiles = x.reshape(P, BLOCK)
assert tuple(tiles.shape) == (2, 4)

for pid in range(P):
    offs = pid * BLOCK + torch.arange(BLOCK)
    mask = offs < N
    assert offs.tolist() == [pid * BLOCK + i for i in range(BLOCK)]
    assert mask.tolist() == [True, True, True, True]
    assert tuple(x[offs][mask].shape) == (4,)
    assert torch.equal(x[offs], tiles[pid])

assert tuple((P, BLOCK)) == (2, 4)

## Worked numerical example

- **Spine.** $x=(1,2,3,4,5,6,7,8)$ and $y=10x=(10,20,30,40,50,60,70,80)$.
- **Grid.** $N=8$, $B=\texttt{BLOCK}=4$, $P=\lceil N/B\rceil=2$.
- **Tile map.**

| pid | offs | mask | x tile | y tile | out tile |
|---:|---|---|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 1,2,3,4 | 10,20,30,40 | 11,22,33,44 |
| 1 | 4,5,6,7 | T,T,T,T | 5,6,7,8 | 50,60,70,80 | 55,66,77,88 |

- **By hand.** $x_0+y_0=1+10=11$ and $x_7+y_7=8+80=88$.
- **Output.** $\text{out}=x+y=11x=(11,22,33,44,55,66,77,88)$.
- **Tail.** For $N=10$, $P=3$; program 2 has $\text{offs}_2=(8,9,10,11)$ and $\text{mask}_2=(T,T,F,F)$.
- **Check.** The next cell asserts every shape, dtype, stride, mask, and value above; nothing is printed.

In [ ]:
N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)
y = 10.0 * x
out = x + y

assert tuple(x.shape) == (8,)
assert tuple(y.shape) == (8,)
assert x.dtype == torch.float32
assert y.dtype == torch.float32
assert out.dtype == torch.float32
assert x.stride() == (1,)
assert y.stride() == (1,)
assert out.stride() == (1,)
assert triton.cdiv(N, BLOCK) == 2
assert out.tolist() == [11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0]
assert torch.equal(out, 11.0 * x)

for pid in range(2):
    offs = pid * BLOCK + torch.arange(BLOCK)
    mask = offs < N
    assert mask.tolist() == [True, True, True, True]
    assert torch.equal(
        out[offs],
        11.0 * torch.arange(1 + pid * BLOCK, 1 + (pid + 1) * BLOCK, dtype=torch.float32),
    )

N2 = 10
x2 = torch.arange(1, N2 + 1, dtype=torch.float32)
y2 = 10.0 * x2
P2 = triton.cdiv(N2, BLOCK)
offs2 = 2 * BLOCK + torch.arange(BLOCK)
mask2 = offs2 < N2

assert P2 == 3
assert tuple((x2 + y2).shape) == (10,)
assert offs2.tolist() == [8, 9, 10, 11]
assert mask2.dtype == torch.bool
assert mask2.tolist() == [True, True, False, False]
assert offs2[mask2].tolist() == [8, 9]
assert (x2 + y2).tolist() == [11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0, 99.0, 110.0]

In [ ]:
def test_worked_example():
    '''Check every shape, stride, mask, and value asserted in Part I.'''
    N, BLOCK = 8, 4
    x = torch.arange(1, N + 1, dtype=torch.float32)
    y = 10.0 * x

    assert tuple(x.shape) == (8,)
    assert x.stride() == (1,)
    assert y.dtype == torch.float32
    assert y.stride() == (1,)
    assert tuple(y.shape) == (8,)
    assert (triton.cdiv(N, BLOCK),) == (2,)

    offs0 = 0 * BLOCK + torch.arange(BLOCK)
    offs1 = 1 * BLOCK + torch.arange(BLOCK)
    assert offs0.tolist() == [0, 1, 2, 3]
    assert offs1.tolist() == [4, 5, 6, 7]
    assert (offs0 < N).tolist() == [True, True, True, True]
    assert (offs1 < N).tolist() == [True, True, True, True]

    out = x + y
    assert out.tolist() == [11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0]

    N2, BLOCK2 = 10, 4
    assert triton.cdiv(N2, BLOCK2) == 3
    offs2 = 2 * BLOCK2 + torch.arange(BLOCK2)
    assert offs2.tolist() == [8, 9, 10, 11]
    assert (offs2 < N2).tolist() == [True, True, False, False]

## 1.5 Host code vs device code

- **Connection.** Section 1.4 defined the grid and the program; this section assigns the two sides of the launch.
- **Rule.** Host code is ordinary Python: allocate tensors, choose the grid and `BLOCK`, launch, synchronize, inspect. Device code is the `@triton.jit` body: pointers, runtime scalars, `constexpr` values, and tile math.
- **Formula.** The host does (allocate, grid, launch); the device does load → $f$ → store per program.
- **Example.** On the spine the host allocates `(8,)` and passes `(2,)`; each device program handles a `(4,)` tile.
- **Where.** Most bugs sit on this boundary: wrong length, non-contiguous view, missing tail, or a `constexpr` that should be runtime.
- **Triton.** The kernel receives `x_ptr`, not `x`; it has no `.shape`, so pass `n_elements`.
- **Failure.** Calling `.shape` or `len()` inside a kernel, or trying to allocate a tensor there.
- **Bridge.** Section 1.6 explains how the host's first launch turns into device code.

In [ ]:
N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)
y = 10.0 * x

# HOST: allocate the output and choose the grid.
out = torch.empty_like(x)
grid = (triton.cdiv(N, BLOCK),)
assert tuple(out.shape) == (8,)
assert out.stride() == (1,)
assert grid == (2,)

# DEVICE: one call per program, operating on a tile.
def device_program(pid, x_ptr, y_ptr, out_ptr, n, block):
    offs = pid * block + torch.arange(block)
    mask = offs < n
    xv = torch.where(mask, x_ptr[offs.clamp(max=n - 1)], torch.zeros(()))
    yv = torch.where(mask, y_ptr[offs.clamp(max=n - 1)], torch.zeros(()))
    out_ptr[offs[mask]] = xv[mask] + yv[mask]
    return offs, mask

for pid in range(grid[0]):
    offs, mask = device_program(pid, x, y, out, N, BLOCK)
    assert tuple(offs.shape) == (4,)
    assert mask.tolist() == [True, True, True, True]

assert out.tolist() == [11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0]
assert torch.equal(out, x + y)

## 1.6 JIT compilation with `@triton.jit`

- **Connection.** Section 1.5 separated host launch from device body; this section follows the first launch through compilation.
- **Rule.** The decorator records the function's AST; the first launch specializes on argument types and `constexpr` values, compiles a binary, and caches it. Later launches with the same key reuse it.
- **Formula.** The key is the tuple (pointer types, scalar types, `constexpr` values, architecture).
- **Example.** `BLOCK=4` and `BLOCK=8` produce two binaries; a new `n_elements` reuses the existing one.
- **Where.** One source serves fp32/fp16 and several block sizes without a build step; the first new key pays a compile cost.
- **Triton.** Keep `@triton.jit` functions in a `.py` file or notebook cell; avoid closures over Python values that should be arguments.
- **Failure.** Blaming a slow first iteration on the kernel when it was compilation, or expecting a changed `BLOCK` to hit the cache.
- **Bridge.** Section 1.7 shows the launch syntax that triggers this process.

In [ ]:
@triton.jit
def signature_demo(out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    tl.store(out_ptr + offs, offs, mask=mask)

import inspect

source = getattr(signature_demo, "fn", signature_demo)
sig = inspect.signature(source)

assert type(signature_demo).__name__ in {"JITFunction", "InterpretedFunction"}
assert list(sig.parameters) == ["out_ptr", "n_elements", "BLOCK"]
assert sig.parameters["out_ptr"].annotation is inspect.Parameter.empty
assert sig.parameters["n_elements"].annotation is inspect.Parameter.empty
assert sig.parameters["BLOCK"].annotation is tl.constexpr

## 1.7 Kernel launch syntax

- **Connection.** Section 1.6 described the specialization step; this section shows the call that starts it.
- **Rule.** The launch is `kernel[(P,)](x, y, out, n_elements, BLOCK=BLOCK)`; the first bracket is the grid, with one entry per `program_id` axis.
- **Formula.** For a 1-D tile kernel, $P=\lceil N/B\rceil=\lfloor(N+B-1)/B\rfloor$.
- **Example.** $N=8$, $B=4$ gives grid `(2,)`; $N=9$ gives `(3,)`.
- **Where.** The grid is host data, not a kernel argument; an undersized grid drops the tail, while an oversized grid is safe only because the mask protects the extra lanes.
- **Triton.** `triton.cdiv(N, B)` computes the ceiling; a grid may also be `lambda meta: (triton.cdiv(N, meta["BLOCK"]),)`.
- **Failure.** Passing a tensor instead of a tuple, or using floor division for the grid.
- **Bridge.** Section 1.8 separates the runtime and compile-time parts of this call.

In [ ]:
N, BLOCK = 8, 4
for n, programs in {1: 1, 8: 2, 9: 3, 10: 3, 12: 3}.items():
    assert triton.cdiv(n, BLOCK) == programs
    assert (programs,)[0] == programs

x = torch.arange(1, N + 1, dtype=torch.float32)
out = torch.empty_like(x)
grid = (triton.cdiv(x.numel(), BLOCK),)

assert tuple(out.shape) == (8,)
assert out.numel() == 8
assert grid == (2,)
assert (triton.cdiv(8, 4),) == (2,)

## 1.8 Compile-time vs runtime arguments

- **Connection.** Section 1.7 passed both data and `BLOCK` in one call; this section says which parts change the binary.
- **Rule.** Every kernel argument is runtime or compile-time; runtime arguments change data, `tl.constexpr` arguments change the program.

| Kind | Examples | Can change without recompiling? | Used for |
|---|---|---|---|
| **Runtime** | pointers, `n_elements`, strides, `scale`, `a`, `b` | yes | data-dependent values |
| **Compile-time** (`tl.constexpr`) | `BLOCK`, `MODE`, `ACT`, `num_warps` | no | shapes, unrolling, branch selection |

- **Formula.** A launch recompiles iff its type or `constexpr` key changes.
- **Example.** `n_elements` is runtime; `BLOCK` and `MODE` are `constexpr`.
- **Where.** Making `n_elements` a `constexpr` would compile a binary per input length; making `BLOCK` runtime is impossible because `tl.arange` needs a compile-time length.
- **Triton.** Annotate only values that must be known at compile time; use uppercase names such as `BLOCK`, `BLOCK_M`, `MODE`.
- **Failure.** A scalar that controls a Python `if` or `range()` must be `tl.constexpr`; a plain runtime scalar raises or forces dynamic control flow.
- **Bridge.** Section 1.9 takes the most common compile-time argument, `BLOCK`, and examines it directly.

In [ ]:
import inspect

@triton.jit
def argument_demo(x_ptr, out_ptr, n_elements, scale,
                  MODE: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    if MODE == 0:
        y = x * scale
    else:
        y = tl.maximum(x * scale, 0.0)
    tl.store(out_ptr + offs, y, mask=mask)

source = getattr(argument_demo, "fn", argument_demo)
sig = inspect.signature(source)
kinds = {name: param.annotation is tl.constexpr for name, param in sig.parameters.items()}

assert kinds == {
    "x_ptr": False,
    "out_ptr": False,
    "n_elements": False,
    "scale": False,
    "MODE": True,
    "BLOCK": True,
}
assert not any(kinds[name] for name in ("x_ptr", "out_ptr", "n_elements", "scale"))

## 1.9 `tl.constexpr`

- **Connection.** Section 1.8 called `BLOCK` a compile-time argument; this section explains what that means inside the kernel.
- **Rule.** `tl.constexpr` marks a value known during compilation; it can be used as a Python integer in `tl.arange`, `range`, `if`, and shape computations.
- **Formula.** $\texttt{BLOCK}=4$ gives $P=2$; $\texttt{BLOCK}=8$ gives $P=1$; the two values are two binaries.
- **Example.** `tl.arange(0, BLOCK)` requires a power of two; 4 and 8 both qualify, 6 does not.
- **Where.** The compiler needs the tile length to allocate registers and choose a layout; constants also let `if MODE == 0` emit only one arm.
- **Triton.** Use `triton.next_power_of_2(n)` when the block must cover an arbitrary `n`.
- **Failure.** Passing a tensor or runtime scalar as `BLOCK`, using a non-power-of-two `tl.arange`, or expecting one cache entry after changing `BLOCK`.
- **Bridge.** Section 1.10 assembles every rule into the first working kernel.

In [ ]:
N = 8
for block, programs in ((4, 2), (8, 1)):
    assert triton.cdiv(N, block) == programs
    assert block > 0
    assert block & (block - 1) == 0
    assert triton.next_power_of_2(block) == block

## 1.10 First working Triton kernel

- **Connection.** Sections 1.1–1.9 defined the tile, grid, host/device split, JIT, launch, and `constexpr`; this section combines them.
- **Rule.** The canonical vector add is one masked tile expression: $\text{out}_i=x_i+y_i$ for $i=0,\ldots,N-1$.
- **Example.** The skeleton is

```python
pid  = tl.program_id(0)
offs = pid * BLOCK + tl.arange(0, BLOCK)
mask = offs < n_elements
x = tl.load(x_ptr + offs, mask=mask, other=0.0)
y = tl.load(y_ptr + offs, mask=mask, other=0.0)
tl.store(out_ptr + offs, x + y, mask=mask)
```

- **Where.** It contains every ingredient: program id, offsets, mask, masked load, tile arithmetic, masked store, grid.
- **Triton.** Use `other=0.0` for addition; the neutral element follows the consuming operation, so a product would use `1.0`.
- **Failure.** Omitting either mask, or choosing `other=` so a later reduction sees the wrong value.
- **Bridge.** Part II writes the scalar reference that this kernel is checked against.
- **Where this leads.** [Chapter 2](../ch02_program_execution/02_complete.ipynb) turns the single `pid` axis into 1-D, 2-D, and 3-D grids and maps programs to matrix coordinates.

In [ ]:
@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x + y, mask=mask)


def add(x, y, BLOCK: int = 256):
    x = torch.as_tensor(x, dtype=torch.float32).contiguous()
    y = torch.as_tensor(y, dtype=torch.float32).contiguous()
    assert x.shape == y.shape, f"shape mismatch: {tuple(x.shape)} vs {tuple(y.shape)}"
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), BLOCK),)
    add_kernel[grid](x, y, out, x.numel(), BLOCK=BLOCK)
    return out

N, BLOCK = 8, 4
x = torch.arange(1, N + 1, dtype=torch.float32)
y = 10.0 * x
out = torch.empty_like(x)
grid = (triton.cdiv(N, BLOCK),)

assert tuple(x.shape) == (8,)
assert tuple(y.shape) == (8,)
assert tuple(out.shape) == (8,)
assert out.dtype == torch.float32
assert out.stride() == (1,)
assert grid == (2,)

print("add_kernel: runtime pointers and n_elements, constexpr BLOCK, grid (2,)")

# Part II — Method 01: naive host reference

- **Connection.** Section 1.10 defined the Triton add; this part defines the scalar semantics it must match.
- **Rule.** The reference is deliberately scalar: one Python iteration per element, plus the PyTorch expression `x + y`.
- **Example.** `naive_add_loop` returns the same `(8,)` result as `naive_add_torch` on the spine.
- **Where.** The loop is slow by design; on a GPU it becomes thousands of host-to-device operations, while on the CPU it is a readable definition.
- **Bridge.** Part III wraps the Triton kernel and checks it against this reference.

In [ ]:
def naive_add_loop(x, y):
    x = torch.as_tensor(x, dtype=torch.float32)
    y = torch.as_tensor(y, dtype=torch.float32)
    assert x.shape == y.shape, f"shape mismatch: {tuple(x.shape)} vs {tuple(y.shape)}"
    out = torch.empty_like(x)
    xf, yf, of = x.reshape(-1), y.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        of[i] = xf[i] + yf[i]
    return out


def naive_add_torch(x, y):
    return torch.as_tensor(x, dtype=torch.float32) + torch.as_tensor(y, dtype=torch.float32)

N = 8
x = torch.arange(1, N + 1, dtype=torch.float32)
y = 10.0 * x
loop = naive_add_loop(x, y)
torch_ref = naive_add_torch(x, y)

assert tuple(loop.shape) == (8,)
assert loop.dtype == torch.float32
assert loop.stride() == (1,)
assert torch.equal(loop, torch_ref)
assert loop.tolist() == [11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0]

print("method 01: Python loop and PyTorch reference agree")

In [ ]:
def test_naive_reference():
    torch.manual_seed(0)
    a = torch.randn(37)
    b = torch.randn(37)
    assert torch.allclose(naive_add_loop(a, b), a + b)

    x = torch.arange(1, 9, dtype=torch.float32)
    y = 10.0 * x
    expected = torch.tensor([11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0])
    assert torch.equal(naive_add_loop(x, y), expected)
    assert torch.equal(naive_add_torch(x, y), expected)

# Part III — Method 02: the Triton vector-add kernel

- **Connection.** Part II supplied the scalar reference; this part supplies the tiled kernel and its tile map.
- **Rule.** One grid axis, one tile per program: program $p$ owns $\text{offs}_p=pB+\{0,\ldots,B-1\}$, $\text{mask}_p=\text{offs}_p\lt N$, and $P=\lceil N/B\rceil$.
- **Example.** Spine $N=8$, $B=4$: $P=2$, every lane live.

| pid | offs | mask | x tile | y tile | out tile |
|---:|---|---|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 1,2,3,4 | 10,20,30,40 | 11,22,33,44 |
| 1 | 4,5,6,7 | T,T,T,T | 5,6,7,8 | 50,60,70,80 | 55,66,77,88 |

- **Tail.** $N=10$, $B=4$: $P=3$; program 2 has two live lanes.

| pid | offs | mask | live lanes |
|---:|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 0,1,2,3 |
| 1 | 4,5,6,7 | T,T,T,T | 4,5,6,7 |
| 2 | 8,9,10,11 | T,T,F,F | 8,9 |

- **Where.** The computation runs on all four lanes; only the load and store are guarded.
- **Triton.** Masked lanes use `other=0.0`, so they add zero and their result is never written.
- **Bridge.** Part IV fuses a second operation to show why the tile model pays off.

In [ ]:
def tile_rows(n, block):
    rows = []
    for pid in range(triton.cdiv(n, block)):
        offs = pid * block + torch.arange(block)
        mask = offs < n
        rows.append((pid, offs.tolist(), mask.tolist(), offs[mask].tolist()))
    return rows

rows8 = tile_rows(8, 4)
rows10 = tile_rows(10, 4)

assert rows8 == [
    (0, [0, 1, 2, 3], [True, True, True, True], [0, 1, 2, 3]),
    (1, [4, 5, 6, 7], [True, True, True, True], [4, 5, 6, 7]),
]
assert rows10 == [
    (0, [0, 1, 2, 3], [True, True, True, True], [0, 1, 2, 3]),
    (1, [4, 5, 6, 7], [True, True, True, True], [4, 5, 6, 7]),
    (2, [8, 9, 10, 11], [True, True, False, False], [8, 9]),
]
assert tuple((len(rows8), 4)) == (2, 4)

print("tile map: N=8 and N=10 rows match the text")

In [ ]:
def test_add_kernel():
    require_gpu()
    torch.manual_seed(0)
    n = 37
    x = torch.randn(n, device=DEVICE)
    y = torch.randn(n, device=DEVICE)
    check("add random N=37 BLOCK=8", add(x, y, BLOCK=8), x + y)

    x = torch.arange(1, 11, device=DEVICE, dtype=torch.float32)
    y = 10.0 * x
    check("add tail N=10 BLOCK=4", add(x, y, BLOCK=4), x + y)

    x = torch.arange(1, 9, device=DEVICE, dtype=torch.float32)
    y = 10.0 * x
    expected = torch.tensor([11.0, 22.0, 33.0, 44.0, 55.0, 66.0, 77.0, 88.0], device=DEVICE)
    out = add(x, y, BLOCK=4)
    assert tuple(out.shape) == (8,)
    assert out.dtype == torch.float32
    check("add spine N=8 BLOCK=4", out, expected)

# Part IV — Method 03: fused residual + activation

- **Connection.** Part III wrote one operation per kernel; this part fuses three into one pass.
- **Rule.** A production epilogue fuses an affine map, a residual add, and an optional activation.
- **Formula.** $z_i=a x_i+y_i+b$; $\text{out}_i=z_i$ when `ACT=0`, and $\text{out}_i=\max(z_i,0)$ when `ACT=1`.
- **Example.** On the spine with $a=2$, $b=1$, $y=10x$, $z=12x+1$; the first four values are negative and become zero under ReLU.
- **Where.** One kernel keeps `z` in registers and writes once, removing two intermediate tensors and their traffic.
- **Triton.** `ACT` is a `tl.constexpr`; the `if ACT == 1` branch is resolved at compile time, so the `ACT=0` binary contains no `max`.
- **Applied use.** This is the residual-plus-activation shape at the end of a transformer block.
- **Bridge.** The next section records what the forward saves and what the cache recompiles.

In [ ]:
@triton.jit
def fused_affine_kernel(x_ptr, y_ptr, out_ptr, n_elements, a, b,
                        ACT: tl.constexpr, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    y = tl.load(y_ptr + offs, mask=mask, other=0.0)
    z = a * x + y + b
    if ACT == 1:
        z = tl.maximum(z, 0.0)
    tl.store(out_ptr + offs, z, mask=mask)


def fused_affine(x, y, a=2.0, b=1.0, act=False, BLOCK=256):
    x = torch.as_tensor(x, dtype=torch.float32).contiguous()
    y = torch.as_tensor(y, dtype=torch.float32).contiguous()
    assert x.shape == y.shape, f"shape mismatch: {tuple(x.shape)} vs {tuple(y.shape)}"
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), BLOCK),)
    fused_affine_kernel[grid](x, y, out, x.numel(), a, b,
                              ACT=1 if act else 0, BLOCK=BLOCK)
    return out

N, BLOCK = 8, 4
x = torch.arange(-4, 4, dtype=torch.float32)
y = 10.0 * x
pre = 2.0 * x + y + 1.0
relu = torch.relu(pre)

assert tuple(x.shape) == (8,)
assert tuple(y.shape) == (8,)
assert tuple(pre.shape) == (8,)
assert pre.dtype == torch.float32
assert pre.stride() == (1,)
assert pre.tolist() == [-47.0, -35.0, -23.0, -11.0, 1.0, 13.0, 25.0, 37.0]
assert relu.tolist() == [0.0, 0.0, 0.0, 0.0, 1.0, 13.0, 25.0, 37.0]

print("method 03: fused affine + ReLU values match the text")

In [ ]:
def test_fused_kernel():
    require_gpu()
    x = torch.arange(-4, 4, device=DEVICE, dtype=torch.float32)
    y = 10.0 * x
    pre = 2.0 * x + y + 1.0

    out = fused_affine(x, y, 2.0, 1.0, act=False, BLOCK=4)
    assert tuple(out.shape) == (8,)
    check("fused affine ACT=0", out, pre)

    out = fused_affine(x, y, 2.0, 1.0, act=True, BLOCK=4)
    assert tuple(out.shape) == (8,)
    check("fused affine ACT=1", out, torch.relu(pre))

    # BLOCK=8 is a second specialization and must give the same numbers.
    check("fused BLOCK=8", fused_affine(x, y, 2.0, 1.0, act=True, BLOCK=8), torch.relu(pre))

# What is saved and what is recompiled

## 2.1 The specialization table

- **Connection.** Part IV introduced a second `constexpr` (`ACT`); this section collects every key that changes a binary.
- **Rule.** A launch is cheap only after its key is cached; the key is the argument types plus the `constexpr` values, not tensor contents.

| Change between launches | Recompile? | Why |
|---|---|---|
| pointer values (`x`, `y`, `out`) | no | same pointer type |
| `n_elements` | no | runtime scalar |
| `a`, `b`, `scale` | no | runtime scalar |
| `BLOCK` | **yes** | constexpr used by `tl.arange` |
| `ACT`, `MODE` | **yes** | constexpr controls `if` |
| input dtype (fp32 vs fp16) | **yes** | different pointer type |
| grid size | no | the grid is not in the binary |
| `num_warps`, `num_stages` | **yes** | launch meta-parameters |

- **Where.** Runtime arguments change the data; compile-time arguments change the program.
- **Triton.** If a value appears in a shape, an `arange`, a `range`, or a Python `if` inside the kernel, make it a `constexpr` and count it in the key.
- **Failure.** Changing `BLOCK` or `ACT` and expecting the old binary; the cache misses and a second binary is compiled.

## 2.2 What is saved for backward

- **Connection.** Section 2.1 covered recompilation; this subsection covers the other cache: tensors saved by the forward.
- **Rule.** For `out = x + y`, the backward is $\partial L/\partial x=\partial L/\partial y=g$, so the forward saves nothing.
- **Where.** For the fused ReLU, the forward either saves the pre-activation tile or recomputes it; saving costs one `N`-element buffer, recomputing costs one extra read of `x`, `y`, `a`, `b`.
- **Triton.** The chapter's backward kernel takes the cheapest option for addition: read `g` once and write it twice.

In [ ]:
def specialization_key(arg_types, constexprs):
    return (tuple(arg_types), tuple(sorted(constexprs.items())))


def test_recompile_policy():
    fp32 = ["fp32*", "fp32*", "fp32*", "int32"]
    k1 = specialization_key(fp32, {"BLOCK": 4, "ACT": 0})
    k2 = specialization_key(fp32, {"BLOCK": 4, "ACT": 0})
    k3 = specialization_key(fp32, {"BLOCK": 8, "ACT": 0})
    k4 = specialization_key(fp32, {"BLOCK": 4, "ACT": 1})
    k5 = specialization_key(["fp16*", "fp16*", "fp16*", "int32"], {"BLOCK": 4, "ACT": 0})

    assert k1 == k2
    assert k1 != k3 and k1 != k4 and k1 != k5

print("recompile policy: same key reuses; BLOCK, ACT, or dtype changes recompile")

# Forward and backward

## 3.1 Vector add

- **Connection.** Section 2.2 stated the result; this section derives it and checks it against autograd.
- **Rule.** With `out = x + y` and upstream gradient $g=\partial L/\partial\text{out}$, the input gradients are $\partial L/\partial x_i=g_i$ and $\partial L/\partial y_i=g_i$.
- **Where.** There is no reduction, no mask to save, and no parameter; the backward kernel is two masked stores of the same tile.
- **Triton.** The check below runs the PyTorch autograd result against these formulas.

## 3.2 Fused affine + ReLU

- **Connection.** Section 3.1 handled the linear add; this subsection adds the affine map and a ReLU.
- **Rule.** For $z=a x+y+b$ and $\text{out}=\max(z,0)$, the activation mask is $m_i=[z_i\gt 0]$ and the gradients are $\partial L/\partial z_i=g_i m_i$, $\partial L/\partial x_i=a g_i m_i$, $\partial L/\partial y_i=g_i m_i$.
- **Formula.** The parameter gradients are $\partial L/\partial a=\sum_i g_i m_i x_i$ and $\partial L/\partial b=\sum_i g_i m_i$.
- **Where.** At $z_i=0$ the derivative is not unique; PyTorch's `relu` uses $m_i=0$, and the check follows that convention.
- **Triton.** The forward either saves `m` (one byte per lane if packed) or recomputes it from `z`; both are valid, and the choice is memory versus compute.

In [ ]:
def test_autograd_matches_manual():
    # Vector add: dx = dy = g, nothing saved.
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0], requires_grad=True)
    y = (10.0 * x.detach()).requires_grad_(True)
    out = x + y
    g = torch.arange(1, 9, dtype=torch.float32)
    out.backward(g)

    assert tuple(out.shape) == (8,)
    assert out.dtype == torch.float32
    assert torch.equal(x.grad, g) and torch.equal(y.grad, g)

    # Fused affine + ReLU: dz = g * [pre > 0], dx = a*dz, dy = dz.
    x2 = torch.tensor([-4.0, -3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0], requires_grad=True)
    y2 = (10.0 * x2.detach()).requires_grad_(True)
    a = torch.tensor(2.0, requires_grad=True)
    b = torch.tensor(1.0, requires_grad=True)
    pre = a * x2 + y2 + b
    out2 = torch.relu(pre)
    out2.sum().backward()

    mask = (pre.detach() > 0).float()
    assert tuple(out2.shape) == (8,)
    assert out2.dtype == torch.float32
    assert mask.tolist() == [0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0]
    assert torch.allclose(x2.grad, a.detach() * mask)
    assert torch.allclose(y2.grad, mask)
    assert torch.allclose(a.grad, (mask * x2.detach()).sum())
    assert torch.allclose(b.grad, mask.sum())

print("autograd: vector add and fused ReLU match the manual formulas")

In [ ]:
@triton.jit
def add_backward_kernel(g_ptr, dx_ptr, dy_ptr, n_elements, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n_elements
    g = tl.load(g_ptr + offs, mask=mask, other=0.0)
    tl.store(dx_ptr + offs, g, mask=mask)
    tl.store(dy_ptr + offs, g, mask=mask)


def add_backward(g, BLOCK: int = 256):
    g = torch.as_tensor(g, dtype=torch.float32).contiguous()
    dx = torch.empty_like(g)
    dy = torch.empty_like(g)
    grid = (triton.cdiv(g.numel(), BLOCK),)
    add_backward_kernel[grid](g, dx, dy, g.numel(), BLOCK=BLOCK)
    return dx, dy


def test_add_backward_kernel():
    require_gpu()
    g = torch.arange(1, 11, device=DEVICE, dtype=torch.float32)
    dx, dy = add_backward(g, BLOCK=4)
    assert tuple(dx.shape) == (10,)
    assert tuple(dy.shape) == (10,)
    assert dx.dtype == torch.float32
    assert dy.dtype == torch.float32
    check("add backward dx", dx, g)
    check("add backward dy", dy, g)

# Pitfalls, checks, and summary

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| grid uses floor division | the tail is never launched | `triton.cdiv(N, BLOCK)` |
| no store mask | out-of-bounds writes | `mask = offs < n_elements` on every store |
| no load mask | out-of-bounds reads | mask every load; choose `other=` |
| `BLOCK` not a power of two | `tl.arange` rejects the length | `triton.next_power_of_2(n)` |
| `x.shape` inside a kernel | the kernel sees only a pointer | pass `n_elements` |
| changing `BLOCK` and expecting a cache hit | a second binary is compiled | treat `BLOCK` as part of the design |
| passing a tensor as the grid | launch-time type error | the grid is a tuple of ints |
| `other=` is not the neutral element | wrong reduction values | `0` for sum, `1` for product, `-inf` for max |
| Python `if` on a runtime scalar | dynamic control flow or an error | make the flag a `tl.constexpr` |
| assuming program order | races between programs | programs are independent; use atomics for shared writes |
| passing a non-contiguous view | wrong addresses | `.contiguous()` on the host, or pass strides |

## Self-check

1. For $N=1000$ and `BLOCK=256`, what is $P$, and which lanes of the last program are live?
2. Write the tile map for $N=10$, `BLOCK=4`: `pid`, `offs`, `mask`, live lanes.
3. Which arguments of `add_kernel` are runtime and which is `tl.constexpr`?
4. What recompiles when `N` changes? What recompiles when `BLOCK` changes?
5. Why must both the load and the store be masked?
6. In `add`, which lines are host code and which line launches device code?
7. What does `other=0.0` mean on a masked lane, and when would `other=1.0` be correct?
8. Why does `tl.arange` require a power-of-two length?

## Summary

- Triton is a tile-level GPU language embedded in Python.
- A launch creates a grid of independent programs.
- Each program computes `pid*BLOCK + arange(BLOCK)`, masks the lanes that do not exist, loads a tile, applies one expression, and stores the tile.
- The host chooses tensors and the grid; the device sees pointers, runtime scalars, and compile-time constants.
- `@triton.jit` specializes and caches a binary per `(types, constexprs)` key.
- Every later chapter reuses the same skeleton with a different expression, offset formula, or grid.

**Where this leads.** [Chapter 2](../ch02_program_execution/02_complete.ipynb) expands the grid to 1-D, 2-D, and 3-D and maps programs to matrix coordinates.

## Exercises

1. **Change the block.** Rerun the spine with `BLOCK=8`. Recompute $P$, the offsets, and the mask. Which program owns lane 7?
2. **Edge case.** Run `add` on $N=1$ with `BLOCK=4`. Which lanes are masked, and what does the output contain?
3. **New expression.** Change `x + y` to `x - y` and update the reference. Which lines of the kernel change?
4. **Runtime scalar.** Add a `scale` argument so the kernel computes `scale * (x + y)`. Is `scale` runtime or `tl.constexpr`? Why?
5. **Compile-time flag.** Add `SQUARE: tl.constexpr` to the add kernel. How many binaries do `SQUARE=0` and `SQUARE=1` produce?
6. **Specialization count.** For `BLOCK` in `{4, 8}` and `ACT` in `{0, 1}`, how many distinct fused binaries does the notebook's kernel need? List the keys.
7. **Tail bug.** Remove the store mask, run $N=10$, and explain the failure. Then restore it and remove only the load mask.
8. **Non-contiguous input.** Pass `x = base[::2]` to `add`. Where does the wrapper make it safe, and what does that cost?
9. **Two dimensions.** Sketch a 2-D grid for a matrix of shape `(M, N)` with tiles `(BLOCK_M, BLOCK_N)`. What are `P_m`, `P_n`, and the two offset vectors? (Chapter 2 answers this.)
10. **Backward traffic.** For the fused kernel, compare saving the ReLU mask with recomputing it: how many extra bytes does each option read/write per lane?

Hints are in [`../../docs/SOLUTIONS.md`](../../docs/SOLUTIONS.md).

In [ ]:
print("Running CPU tests ...")
test_worked_example()
test_naive_reference()
test_recompile_policy()
test_autograd_matches_manual()

print()
print("Running GPU tests (skipped without CUDA/ROCm) ...")
for fn in (
    test_add_kernel,
    test_fused_kernel,
    test_add_backward_kernel,
):
    run_if_gpu(fn)